In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType,FloatType, DateType
import pyspark.sql.functions as F

In [0]:
raw_path = '/Volumes/f1_racing/raw_data/raw_files/'

### Drivers

In [0]:
drivers_schema = StructType([
    StructField('driverId', IntegerType(),False),
    StructField('driverRef',StringType(),True),
    StructField('number',StringType(),True),
    StructField('code',StringType(),True),
    StructField('forename',StringType(),True),
    StructField('surname',StringType(),True),
    StructField('dob',DateType(),True),
    StructField('nationality',StringType(),True),
    StructField('url',StringType(),True),
])

In [0]:
df_drivers = spark.read.csv(path=f'{raw_path}/drivers.csv',header=True, inferSchema=True, schema=drivers_schema)

df_drivers = df_drivers.withColumn('source_fie',F.col('_metadata.file_path'))\
                        .withColumn('ingested_at',F.current_timestamp())

display(df_drivers.limit(5))

In [0]:
df_drivers.write.format('delta')\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable('f1_racing.bronze.brz_drivers')

### Circuits

In [0]:
df_circuits = spark.read.csv(path=f'{raw_path}circuits.csv',inferSchema=True,header=True)

display(df_circuits.limit(5))

In [0]:
circuits_schema = StructType([
    StructField('circuitId', IntegerType(),False),
    StructField('circuitRef',StringType(),True),
    StructField('name',StringType(),True),
    StructField('location',StringType(),True),
    StructField('country',StringType(),True),
    StructField('lat',FloatType(),True),
    StructField('lng',FloatType(),True),
    StructField('alt',IntegerType(),True),
    StructField('url',StringType(),True),
])

In [0]:
df_circuits = spark.read.csv(path=f'{raw_path}circuits.csv',inferSchema=True,header=True,schema=circuits_schema)

df_circuits = df_circuits.withColumn('source_fie',F.col('_metadata.file_path'))\
                        .withColumn('ingested_at',F.current_timestamp())

display(df_circuits.limit(5))

In [0]:
df_circuits.write.format('delta')\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable('f1_racing.bronze.brz_circuits')

### Races

In [0]:
df_races = spark.read.csv(path=f'{raw_path}races.csv',inferSchema=True,header=True)

display(df_races)

In [0]:
df_races = df_races.withColumn('source_fie',F.col('_metadata.file_path'))\
                        .withColumn('ingested_at',F.current_timestamp())

display(df_races.limit(4))


In [0]:
df_races.write.format('delta')\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable('f1_racing.bronze.brz_races')

### Results

In [0]:
df_results = spark.read.csv(path=f'{raw_path}results.csv',inferSchema=True,header=True)
df_results = df_results.withColumn('source_file',F.col('_metadata.file_path'))\
                        .withColumn('ingested_at',F.current_timestamp())
display(df_results.limit(4))

In [0]:
df_results = df_results.withColumn(
    'number', F.when(
        F.col('number') == r'\N', None
    ).otherwise(
        F.col('number')
    ))\
    .withColumn(
    'grid', F.when(
        F.col('grid') == r'\N', None
    ).otherwise(
        F.col('grid')
    ))\
    .withColumn(
    'position', F.when(
        F.col('position') == r'\N', None
    ).otherwise(
        F.col('position')
    ))\
    .withColumn(
    'positionText', F.when(
        F.col('positionText') == r'\N', None
    ).otherwise(
        F.col('positionText')
    ))\
    .withColumn(
    'time', F.when(
        F.col('time') == r'\N', None
    ).otherwise(
        F.col('time')
    ))\
    .withColumn(
    'milliseconds', F.when(
        F.col('milliseconds') == r'\N', None
    ).otherwise(
        F.col('milliseconds')
    ))\
    .withColumn(
    'fastestLap', F.when(
        F.col('fastestLap') == r'\N', None
    ).otherwise(
        F.col('fastestLap')
    ))\
    .withColumn(
    'rank', F.when(
        F.col('rank') == r'\N', None
    ).otherwise(
        F.col('rank')
    ))\
    .withColumn(
    'fastestLapSpeed', F.when(
        F.col('fastestLapSpeed') == r'\N', None
    ).otherwise(
        F.col('fastestLapSpeed')
    ))\
    .withColumn(
    'fastestLapTime', F.when(
        F.col('fastestLapTime') == r'\N', None
    ).otherwise(
        F.col('fastestLapTime')
    ))

In [0]:
display(df_results.limit(4))

In [0]:
df_results.printSchema()

In [0]:
import pyspark.sql.functions as F

In [0]:
df_results.write.format('csv')\
    .option('header', True)\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .save('/Volumes/f1_racing/raw_data/raw_files/results_with_null/')


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

results_schema = StructType([
    StructField("resultId", IntegerType(), True),
    StructField("raceId", IntegerType(), True),
    StructField("driverId", IntegerType(), True),
    StructField("constructorId", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("grid", IntegerType(), True),
    StructField("position", StringType(), True),
    StructField("positionText", StringType(), True),
    StructField("positionOrder", IntegerType(), True),
    StructField("points", DoubleType(), True),
    StructField("laps", FloatType(), True),
    StructField("time", StringType(), True),
    StructField("milliseconds", IntegerType(), True),
    StructField("fastestLap", StringType(), True),
    StructField("rank", StringType(), True),
    StructField("fastestLapTime", StringType(), True),
    StructField("fastestLapSpeed", FloatType(), True),
    StructField("statusId", IntegerType(), True),
    StructField("source_file", StringType(), False),
    StructField("ingested_at", TimestampType(), False)
])

In [0]:
df_results = spark.read.csv('/Volumes/f1_racing/raw_data/raw_files/results_with_null/',header=True, inferSchema=True,schema= results_schema)

display(df_results)

In [0]:
%sql
drop table f1_racing.bronze.brz_results

In [0]:
df_results.write.format('delta')\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable('f1_racing.bronze.brz_results')

### Status

In [0]:
df_status = spark.read.csv(path=f'{raw_path}status.csv',inferSchema=True,header=True)

display(df_status.limit(4))

In [0]:
df_status = df_status.withColumn('source_fie',F.col('_metadata.file_path'))\
                        .withColumn('ingested_at',F.current_timestamp())
display(df_status.limit(4))

In [0]:
df_status.write.format('delta')\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable('f1_racing.bronze.brz_status')